# Phase 2: the VLM evaluation run

One Colab session produces every VLM number Phase 2 needs. The output is
`vlm_outputs.json`, which you replay locally through the same scoring path the
rules baseline goes through.

**Runtime > Change runtime type > T4 GPU** before running anything.

100 documents: 50 from `gst_in_synthetic` (layouts we wrote) and 50 from
`fatura2` (50 templates designed by other people). Both, deliberately. The
rules baseline scores 1.000 on the first and 0.453 on the second, and the whole
question is which side of that gap a 2B VLM lands on.

This notebook does **not** score anything. Coercion, validation and metrics all
happen locally with the frozen Phase 0 harness, so a difference in the results
table is a difference in the model and not in post-processing.

Expect 30-60 minutes. It checkpoints every 5 documents and resumes, so a
disconnect costs you minutes rather than the run.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ~3 minutes.
!pip -q install "transformers>=4.45" accelerate qwen-vl-utils pillow

import transformers
print("transformers", transformers.__version__)

## 1. Get the documents

Cloned from the public repo, so there is nothing to upload.

In [ ]:
!git clone --depth 1 -b development -q https://github.com/jayani-06/invoice-extract-vlm.git repo

import collections
import json
import zipfile

with zipfile.ZipFile("repo/vlm_eval_payload.zip") as z:
    z.extractall("payload")

manifest = json.load(open("payload/manifest.json"))
docs = manifest["documents"]

print(f"{len(docs)} documents")
print("model :", manifest["model_id"])
print("mode  :", "image + OCR text" if manifest.get("ocr_hint") else "vision only")
print("image :", manifest["profile"])
print("by corpus:", dict(collections.Counter(d["corpus"] for d in docs)))
print("by split :", dict(collections.Counter(d.get("split") for d in docs)))

## 2. Check the prompt before spending the GPU

The previous run on this project was thrown away after the fact. Its prompt
template carried a concrete example amount (`123.45`) to show the shape of the
`tax_lines.amount` field, and Qwen2-VL-2B copied that value verbatim in 8 of 8
responses -- corrupting the one field the example was meant to illustrate.

So this is checked **before** the model loads rather than after the results come
back: every value in the template must be null, and no numeral may appear
anywhere in the prompt for the model to copy.

In [ ]:
prompt = docs[0]["prompt"]

# A plain digit test rather than a number regex: the failure mode is the model
# copying ANY digit out of the template, so the strictest possible check is
# also the simplest one to read.
digits = sorted({c for c in prompt if c.isdigit()})
assert not digits, f"prompt contains copyable digits: {digits}"

# In vision-only mode every document must get a byte-identical prompt. If they
# differ, something is building prompts per document and the run is not a
# controlled comparison.
if not manifest.get("ocr_hint"):
    assert len({d["prompt"] for d in docs}) == 1, "prompts differ across documents"

print("prompt is clean:", len(prompt), "chars, no digits")
print(prompt[-400:])

## 3. Load the model

Qwen2-VL-2B in fp16. It fits a T4 comfortably, which is why the compute plan
picks it as the first baseline rather than a 7B in 4-bit.

In [ ]:
import torch
from transformers import AutoProcessor

# transformers v5 renamed two things at once, and both only fail on a GPU box:
#   AutoModelForVision2Seq -> AutoModelForImageTextToText   (old name removed)
#   from_pretrained(torch_dtype=...) -> from_pretrained(dtype=...)
# Probe for each rather than pinning, so this runs on whatever Colab ships.
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM

MODEL_ID = manifest["model_id"]
try:
    model = AutoVLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
except TypeError:
    model = AutoVLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()

print(MODEL_ID, "loaded on", model.device)
print(f"{torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

## 4. Run every document

A model that returns unparseable text is a **result**, not a crash: it scores as
a fully missed document. Failures are recorded and the loop continues, and the
denominator stays at 100 either way.

Greedy decoding (`do_sample=False`), so the run is reproducible.

Checkpointed to `vlm_outputs.json` every 5 documents. Re-running this cell after
a disconnect picks up where it stopped.

In [ ]:
import pathlib
import time

from PIL import Image

SYSTEM_PROMPT = manifest["system_prompt"]
MAX_NEW_TOKENS = manifest.get("max_new_tokens", 1536)
OUT = pathlib.Path("vlm_outputs.json")

# Resume: keep whatever a previous attempt already produced.
outputs = {}
if OUT.exists():
    outputs = (json.loads(OUT.read_text()) or {}).get("outputs", {})
    print(f"resuming with {len(outputs)} documents already done")


def save():
    OUT.write_text(
        json.dumps(
            {
                "model": MODEL_ID.split("/")[-1],
                "model_id": MODEL_ID,
                "preset": manifest.get("preset"),
                "ocr_hint": manifest.get("ocr_hint"),
                "profile": manifest["profile"],
                "max_new_tokens": MAX_NEW_TOKENS,
                "transformers_version": transformers.__version__,
                "gpu": torch.cuda.get_device_name(0),
                "decoding": "greedy",
                "outputs": outputs,
            },
            indent=1,
        )
    )


pending = [d for d in docs if d["doc_id"] not in outputs]
started = time.time()

for i, entry in enumerate(pending, 1):
    doc_id = entry["doc_id"]
    try:
        image = Image.open(pathlib.Path("payload") / entry["image"]).convert("RGB")
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": entry["prompt"]},
                ],
            },
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

        t0 = time.perf_counter()
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        elapsed_ms = (time.perf_counter() - t0) * 1000

        reply = processor.decode(
            generated[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
        )
        outputs[doc_id] = {
            "raw_response": reply,
            "elapsed_ms": round(elapsed_ms, 1),
            "error": None,
        }
    except Exception as exc:
        # Recorded, not raised: an OOM or a decode failure on one page must not
        # cost the other 99.
        outputs[doc_id] = {
            "raw_response": "",
            "elapsed_ms": 0.0,
            "error": f"{type(exc).__name__}: {exc}",
        }
        print(f"  {doc_id}: {outputs[doc_id]['error']}")

    if i % 5 == 0 or i == len(pending):
        save()
        rate = (time.time() - started) / i
        left = rate * (len(pending) - i) / 60
        print(f"  {i}/{len(pending)}  {rate:.1f}s/doc  ~{left:.0f} min left")

save()
print(f"\ndone: {len(outputs)}/{len(docs)} documents")

## 5. Sanity-check the responses before you leave Colab

Three things are worth knowing now rather than after the download, because each
one means repeating the run:

- **the placeholder is back** -- any response containing `123.45` means the
  prompt regressed
- **a high parse-failure rate** -- if most responses are unparseable, the
  `max_new_tokens` budget is probably truncating the JSON mid-object
- **empty responses** -- a loading or decode problem, not a model result

In [ ]:
import re

_FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)
_TRAILING_COMMA = re.compile(r",(\s*[}\]])")


def _balanced(text):
    start = text.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc:
                esc = False
            elif ch == chr(92):
                esc = True
            elif ch == chr(34):
                in_str = False
            continue
        if ch == chr(34):
            in_str = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start : i + 1]
    return None


def parses(response):
    """Mirrors models/prompting.extract_json_object. Indicative only -- the
    authoritative parse happens locally with the real one."""
    if not response or not response.strip():
        return False
    candidates = [response.strip()]
    fence = _FENCE.search(response)
    if fence:
        candidates.append(fence.group(1).strip())
    bal = _balanced(response)
    if bal:
        candidates.append(bal)
    for c in candidates:
        for attempt in (c, _TRAILING_COMMA.sub(r"\1", c)):
            try:
                obj = json.loads(attempt)
            except Exception:
                continue
            if isinstance(obj, dict):
                return True
    return False


n = len(outputs)
failed = [k for k, v in outputs.items() if not parses(v["raw_response"])]
empty = [k for k, v in outputs.items() if not (v["raw_response"] or "").strip()]
placeholder = [k for k, v in outputs.items() if "123.45" in (v["raw_response"] or "")]
errors = [k for k, v in outputs.items() if v.get("error")]
ms = [v["elapsed_ms"] for v in outputs.values() if v["elapsed_ms"]]

print(f"documents      : {n}")
print(f"parse failures : {len(failed)}  ({len(failed) / max(n, 1):.1%})")
print(f"empty replies  : {len(empty)}")
print(f"hard errors    : {len(errors)}")
print(f"mean ms/doc    : {sum(ms) / max(len(ms), 1):.0f}")
print()
if placeholder:
    print(f"*** {len(placeholder)} response(s) contain 123.45 -- the prompt has regressed.")
    print("*** Do not use this run. Fix models/prompting.py and rebuild the payload.")
else:
    print("no placeholder copying")

if failed:
    print(f"first unparseable response ({failed[0]}):")
    print(outputs[failed[0]]["raw_response"][:600])

In [ ]:
from google.colab import files

files.download("vlm_outputs.json")

## 6. Back on your machine

Replay the responses through the frozen scoring path -- the same code, the same
post-processing and the same metrics the rules baseline gets:

```bash
python scripts/run_extraction.py --corpus data/processed/gst_in_synthetic \\
    --extractor vlm:qwen2-vl-2b --replay vlm_outputs.json \\
    --doc-ids vlm_eval_payload.selection.json --out reports/phase2/vlm_gst

python scripts/run_extraction.py --corpus data/processed/fatura2 \\
    --extractor vlm:qwen2-vl-2b --replay vlm_outputs.json \\
    --doc-ids vlm_eval_payload.selection.json --out reports/phase2/vlm_fatura2

python -m invoice_extract.eval.harness \\
    --gold reports/phase2/vlm_gst/gold.jsonl --pred reports/phase2/vlm_gst/pred.jsonl
```

Then merge into the demo so screen 4 shows a genuine rules-vs-model comparison:

```bash
python scripts/build_demo_cache.py --vlm vlm_outputs.json
```
